## Notebook showing the general numbers of the MPRA dataset
- Loads sequencing replicates
- Filters with thresholds
- Merge replicates 

In [2]:
import pandas as pd
import numpy as np
import os
import yaml
import re


# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

_total_number_oligos = config["analysis_info"]["total_number_oligos"]

### Check MPRAsnakeflow results and merge the filtered replicates based on the assignment file
#### Investigate assignment file 

In [3]:
def get_assignment_ratio(assignment_numbers):
    """
    Gets assignment numbers from the assignment file (e.g. 9/11) and returns their ratio 0.82
    """
    assigned_barcodes = int(assignment_numbers.split("/")[0])
    found_barcodes = int(assignment_numbers.split("/")[1])
    return round(assigned_barcodes/found_barcodes, 4)

def get_found_barcode_number(assignment_numbers):
    """
    Gets assignment numbers from the assignment file (e.g. 9/11) and returns their ratio 0.82
    """
    found_barcodes = int(assignment_numbers.split("/")[1])
    return found_barcodes

def get_assignment_barcode_number(assignment_numbers):
    """
    Gets assignment numbers from the assignment file (e.g. 9/11) and returns their ratio 0.82
    """
    assigned_barcodes = int(assignment_numbers.split("/")[0])
    return assigned_barcodes

# combine mprasnakeflow result
def combine_replicates(df_allreps, total_dna_counts, total_rna_counts):
    df_allreps = df_allreps.groupby(by=["condition", "name"]).aggregate(
        {
            "replicate": "count",
            "dna_counts": ["sum", "mean"],
            "rna_counts": ["sum", "mean"],
            "dna_normalized": "mean",
            "rna_normalized": "mean",
            "ratio": "mean",
            "log2": "mean",
            "n_obs_bc": ["sum", "mean"],
        }
    )

    df_allreps = df_allreps.reset_index()
    df_out = df_allreps.iloc[:, 0:2]
    df_out.columns = ["condition", "name"]

    df_out["replicates"] = df_allreps.replicate["count"]

    scaling = 10**6

    df_out["dna_counts"] = df_allreps.dna_counts["sum"]
    df_out["rna_counts"] = df_allreps.rna_counts["sum"]


    df_out["dna_normalized"] = df_out["dna_counts"] / total_dna_counts * scaling
    df_out["rna_normalized"] = df_out["rna_counts"] / total_rna_counts * scaling

    df_out["ratio"] = df_out["rna_normalized"] / df_out["dna_normalized"]
    df_out["log2"] = np.log2(df_out.ratio)

    df_out["mean_dna_counts"] = df_allreps.dna_counts["mean"]
    df_out["mean_rna_counts"] = df_allreps.rna_counts["mean"]
    df_out["mean_dna_normalized"] = df_allreps.dna_normalized["mean"]
    df_out["mean_rna_normalized"] = df_allreps.rna_normalized["mean"]
    df_out["mean_ratio"] = df_allreps.ratio["mean"]
    df_out["mean_log2"] = df_allreps.log2["mean"]

    df_out["mean_n_obs_bc"] = df_allreps.n_obs_bc["mean"].apply(int)
    return df_out

In [4]:
# load assignemnt file
assignemnt_file = pd.read_csv(config['files']['final_design']['assignment_file'], sep="\t", header=None)
assignemnt_file.columns = ['barcode', 'name', 'alignment_info', 'assignment_number']

In [5]:
assignemnt_file = assignemnt_file.assign(assignment_ratio = assignemnt_file['assignment_number'].apply(get_assignment_ratio))
assignemnt_file = assignemnt_file.assign(assigned_barcodes_number = assignemnt_file['assignment_number'].apply(get_assignment_barcode_number))
assignemnt_file = assignemnt_file.assign(found_barcodes_number = assignemnt_file['assignment_number'].apply(get_found_barcode_number))

In [6]:
print(f"Number of barcodes assigned: {assignemnt_file.shape[0]}")

Number of barcodes assigned: 6305466


In [7]:
assignemnt_file.head()

,barcode,name,alignment_info,assignment_number,assignment_ratio,assigned_barcodes_number,found_barcodes_number
0,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,15;270M;NM:i:0;MD:Z:270;60,5/5,1.0000,5,5
1,AAAAAAAAAACAAGT,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,6/7,0.8571,6,7
2,AAAAAAAAAACACCA,cardiac_neuro_cava_random:ALT_FTO|ENSG00000140...,15;270M;NM:i:0;MD:Z:270;6,9/10,0.9000,9,10
3,AAAAAAAAAACCTCG,cardiac_neuro_cava_random:REF_FKRP|ENSG0000018...,15;270M;NM:i:0;MD:Z:270;6,7/7,1.0000,7,7
4,AAAAAAAAAAGCTGG,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,7/8,0.8750,7,8


In [8]:
print(f"Minimal number of assignment_ratio: {assignemnt_file['assignment_ratio'].min()}") # 0.7
print(f"Minimal number of assigned_barcodes_number: {assignemnt_file['assigned_barcodes_number'].min()}") # 3
print(f"Minimal number of found_barcodes_number: {assignemnt_file['found_barcodes_number'].min()}") # 3

Minimal number of assignment_ratio: 0.7
Minimal number of assigned_barcodes_number: 3
Minimal number of found_barcodes_number: 3


#### Merge Assignment files with the replicates
1. merge with assignment file
2. compute n_obs for each replicate 

In [9]:
# load replicate files

# example workflow for one replicate
replicate_1 = pd.read_csv(config['files']['final_design']['barcodes_rep1'], sep="\t", header=None)
replicate_1.columns = ['barcode', 'dna_counts', 'rna_counts']
replicate_1['replicate'] = 1

print(f"Number of barcodes in replicate 1: {replicate_1['barcode'].nunique()}")
# print(f"Number of barcodes in replicate 1: {replicate_1.shape[0]}")

# same for replicate 2
replicate_2 = pd.read_csv(config['files']['final_design']['barcodes_rep2'], sep="\t", header=None)
replicate_2.columns = ['barcode', 'dna_counts', 'rna_counts']
replicate_2['replicate'] = 2

print(f"Number of barcodes in replicate 2: {replicate_2['barcode'].nunique()}")
# print(f"Number of barcodes in replicate 2: {replicate_2.shape[0]}")

# and 3
replicate_3 = pd.read_csv(config['files']['final_design']['barcodes_rep3'], sep="\t", header=None)
replicate_3.columns = ['barcode', 'dna_counts', 'rna_counts']
replicate_3['replicate'] = 3

print(f"Number of barcodes in replicate 3: {replicate_3['barcode'].nunique()}")
# print(f"Number of barcodes in replicate 3: {replicate_3.shape[0]}")

# Number of barcodes in replicate 1: 4695996
# Number of barcodes in replicate 2: 4704637
# Number of barcodes in replicate 3: 4734536

Number of barcodes in replicate 1: 4695996
Number of barcodes in replicate 2: 4704637
Number of barcodes in replicate 3: 4734536


In [10]:
# merge with assignment file
replicate_1_assignment = pd.merge(replicate_1, assignemnt_file[['barcode', 'name', 'assignment_number']], on='barcode', how='left')

# merge with assignment file
replicate_2_assignment = pd.merge(replicate_2, assignemnt_file[['barcode', 'name', 'assignment_number']], on='barcode', how='left')

# merge with assignment file
replicate_3_assignment = pd.merge(replicate_3, assignemnt_file[['barcode', 'name', 'assignment_number']], on='barcode', how='left')

In [11]:
replicate_1_assignment.loc[replicate_1_assignment['assignment_number'].isna()] # all can be matched

,barcode,dna_counts,rna_counts,replicate,name,assignment_number


In [12]:
replicate_1_assignment.loc[replicate_1_assignment['name'] == 'cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000181722.18|EH38E2228539_rev_tile1-1_ZBTB20|ENSG00000181722.18|EH38E2228539|3-114510161-A-G'].shape[0]

45

: 

In [56]:
# compute n_obs: groupby name and count the number of barcodes
replicate_1_assignment['n_obs_bc'] = replicate_1_assignment.groupby('name')['barcode'].transform('count')

In [57]:
replicate_1_assignment.loc[replicate_1_assignment['name'] == 'cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000181722.18|EH38E2228539_rev_tile1-1_ZBTB20|ENSG00000181722.18|EH38E2228539|3-114510161-A-G']

,barcode,dna_counts,rna_counts,replicate,name,assignment_number,n_obs_bc
1,AAAAAAAAAACAAGT,1,1,1,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,6/7,45
1876,AAAAACTATCGCCTA,3,1,1,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,22/23,45
27289,AAACTACTAGCGGAT,2,2,1,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,4/4,45
55420,AAATCTGTTTTGTCA,1,1,1,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,6/7,45
148447,AAGCCCCTATATGTT,4,5,1,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,23/24,45
168281,AAGGCGTGGTCGTTA,4,11,1,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,11/15,45
575334,AGAGGCGAACCTATT,8,6,1,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,42/44,45
635666,AGCCTTGAGAAACGA,4,9,1,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,49/57,45
733715,AGGGGGTCCCTCGTT,4,4,1,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,24/25,45
996572,ATGAGTGTCCCGGAA,8,7,1,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,47/50,45


#### Investigate number of barcodes per replicate
- What is the average number of inserts? 
- add n_obs_bc 
- drop duplicates in name 
- histogram of n_obs_bc + mean and median of n_obs_bc

In [58]:
replicate_1_assignment['n_obs_bc'] = replicate_1_assignment.groupby('name')['barcode'].transform('count')
replicate_2_assignment['n_obs_bc'] = replicate_2_assignment.groupby('name')['barcode'].transform('count')
replicate_3_assignment['n_obs_bc'] = replicate_3_assignment.groupby('name')['barcode'].transform('count')

In [59]:

print('Replicate 1')
print(f"Number of oligos assigned: {replicate_1_assignment['name'].nunique()}") # 74782
print(f"Minimal number of n_obs_bc: {replicate_1_assignment['n_obs_bc'].min()}") # 1

print('Replicate 2')
print(f"Number of oligos assigned: {replicate_2_assignment['name'].nunique()}") # 74760
print(f"Minimal number of n_obs_bc: {replicate_2_assignment['n_obs_bc'].min()}") # 1

print('Replicate 3')
print(f"Number of oligos assigned: {replicate_3_assignment['name'].nunique()}") # 74781
print(f"Minimal number of n_obs_bc: {replicate_3_assignment['n_obs_bc'].min()}") # 1

Replicate 1
Number of oligos assigned: 74782
Minimal number of n_obs_bc: 1
Replicate 2
Number of oligos assigned: 74760
Minimal number of n_obs_bc: 1
Replicate 3
Number of oligos assigned: 74781
Minimal number of n_obs_bc: 1


#### Investigate the overlab and union without filtering: 
- How many sequences are in all replicates?
- Overlap of sequences between the replicates is `74597`
- All replicates together provide measurements for `74907` sequences
- 7% der oligos können wir nicht analysieren, weil wir sie nicht assignen können


In [60]:
rep1_oligos = set(replicate_1_assignment['name'].to_list())
rep2_oligos = set(replicate_2_assignment['name'].to_list())
rep3_oligos = set(replicate_3_assignment['name'].to_list())

#  Check the overlap between replicates
replicate_overlap = rep1_oligos.intersection(rep2_oligos).intersection(rep3_oligos)
print(f'Before filtering based on barcode count: {len(replicate_overlap)} oligos overlap') # 74597
#  Check the overlap between replicates
replicate_union = rep1_oligos.union(rep2_oligos).union(rep3_oligos)
print(f'Before filtering based on barcode count: {len(replicate_union)} oligos union') # 74907

# Check how many get lost through experiment and MPRAsnakeflow:
print(f'Intersection based number of oligos lost through experiment and MPRAsnakeflow: {_total_number_oligos - len(replicate_overlap)} ({round((_total_number_oligos - len(replicate_overlap))/_total_number_oligos*100, 3)}%)')
print(f'Union based number of oligos lost through experiment and MPRAsnakeflow: {_total_number_oligos - len(replicate_union)} ({round((_total_number_oligos - len(replicate_union))/_total_number_oligos*100, 3)}%)')


Before filtering based on barcode count: 74597 oligos overlap
Before filtering based on barcode count: 74907 oligos union
Intersection based number of oligos lost through experiment and MPRAsnakeflow: 5618 (7.004%)
Union based number of oligos lost through experiment and MPRAsnakeflow: 5308 (6.617%)


#### Filter replicates based on barcode count: 
- final resequencing with wrong UMI and not perfect alignment step: 
    - 5 barcodes per oligo per replicate
        - Rep1: 72057
        - Rep2: 72053
        - Rep3: 72043
        - 71627 oligos overlap (0.8929)
        - 72422 oligos in total (union) (0.9028)
    - 10 barcodes per oligo per replicate
        - Rep1: 68657
        - Rep2: 68677
        - Rep3: 68673
        - 67977 oligos overlap (0.8474)
        - 69297 oligos in total (union) (0.8639)
    - 20 barcodes per oligo per replicate
        - Rep1: 61292
        - Rep2: 61320
        - Rep3: 61351
        - 60271 oligos overlap (0.7514)
        - 62313 oligos in total (union) (0.7768)
    - 50 barcodes per oligo per replicate
        - Rep1: 39313
        - Rep2: 39355
        - Rep3: 39498
        - 37977 oligos overlap (0.4734)
        - 40803 oligos in total (union) (0.5087)

In [61]:
def filter_barcodes(df, print_text, threshold=5, barcode_column="n_obs_bc", verbose=True):
    """
    Filters the given df based on the barcode column, prints the number of unique names in df and returns the filtered df
    """
    df_filtered = df.loc[df[barcode_column] >= threshold]
    if verbose:
        print(f"{name}: {df_filtered['name'].nunique()}")
    return df_filtered

: 

In [39]:
print("Start filtering the replicates based on the number of barcodes")

rep_df_dict = {
    'Rep1': replicate_1_assignment,
    'Rep2': replicate_2_assignment,
    'Rep3': replicate_3_assignment
}

for threshold in [5,10,20,50]:
    replicate_oligo_sets = []
    print(f"Minimum number of barcodes: {threshold}")
    for name, df in rep_df_dict.items():
        rep = filter_barcodes(df, print_text=name, threshold=threshold)
        # compute set of oligos per replicate
        replicate_oligo_sets.append(set(rep['name'].to_list()))
    # compute the overlap between replicates
    replicate_overlap = replicate_oligo_sets[0].intersection(replicate_oligo_sets[1]).intersection(replicate_oligo_sets[2])
    print(f'{len(replicate_overlap)} oligos overlap')
    # compute the proportion compared to the overall number of oligos (80215)
    print(f'{round(len(replicate_overlap) / _total_number_oligos, 4)} proportion of oligos overlap')
    # Compute the proportion compared to the maximal possible overlap (union of all replicates without filtering)
    print(f'{round(len(replicate_overlap) / len(replicate_union), 4)} proportion of oligos overlap in the union')
    # compute the union of oligos between replicates
    replicate_union = replicate_oligo_sets[0].union(replicate_oligo_sets[1]).union(replicate_oligo_sets[2])
    print(f'{len(replicate_union)} oligos in total')
    # compute the proportion compared to the overall number of oligos (80215)
    print(f'{round(len(replicate_union) / _total_number_oligos, 4)} proportion of oligos union')

Start filtering the replicates based on the number of barcodes
Minimum number of barcodes: 5
Rep1: 72057
Rep2: 72053
Rep3: 72043
71627 oligos overlap
0.8929 proportion of oligos overlap
72422 oligos in total
0.9028 proportion of oligos union
Minimum number of barcodes: 10
Rep1: 68657
Rep2: 68677
Rep3: 68673
67977 oligos overlap
0.8474 proportion of oligos overlap
69297 oligos in total
0.8639 proportion of oligos union
Minimum number of barcodes: 20
Rep1: 61292
Rep2: 61320
Rep3: 61351
60271 oligos overlap
0.7514 proportion of oligos overlap
62313 oligos in total
0.7768 proportion of oligos union
Minimum number of barcodes: 50
Rep1: 39313
Rep2: 39355
Rep3: 39498
37977 oligos overlap
0.4734 proportion of oligos overlap
40803 oligos in total
0.5087 proportion of oligos union


#### Merge replicates together
- merge all replicates with their barcodes together / looked into mprasnakeflow and Pias commit: (https://github.com/kircherlab/MPRAsnakeflow/commit/2e534d11b3b517cf4bcc6cca6f59dc2b04ca5b77) in `workflow/scripts/count/merge_replicates_barcode_counts.py` the barcode assigned files are combined and converted into bc mpralm readable format

In [36]:
replicate_1_assignment.head()

,barcode,rep1_dna_counts,rep1_rna_counts,name,assignment_number
0,AAAAAAAAAACAAGT,1,1,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,6/7
1,AAAAAAAAAAGCTGG,4,11,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,7/8
2,AAAAAAAAAATCCTA,2,6,cardiac_neuro_cava_random:ALT_ANK3|ENSG0000015...,15/16
3,AAAAAAAAAATCTAC,3,12,cardiac_neuro_cava_random:ALT_NBEA|ENSG0000017...,19/20
4,AAAAAAAAAATGTTT,4,4,cardiac_neuro_cava_random:ALT_CFAP91|ENSG00000...,8/10


In [12]:
replicate_1_assignment['name'].to_list()

['GC_Vista:eye;fb;hb;mb_hs1644_vistaElementControl|chr6:139888452-139888701',
 'cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000181722.18|EH38E2228539_rev_tile1-1_ZBTB20|ENSG00000181722.18|EH38E2228539|3-114510161-A-G',
 'cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000182185.19|EH38E3101266_fwd_tile1-1_RAD51B|ENSG00000182185.19|EH38E3101266|14-68243623-G-A',
 'cardiac_neuro_cava_random:ALT_ANK3|ENSG00000151150.22|EH38E1470921_rev_tile1-1_ANK3|ENSG00000151150.22|EH38E1470921|10-60424168-C-G',
 'cardiac_neuro_cava_random:ALT_NBEA|ENSG00000172915.20|EH38E1667609_fwd_tile1-1_NBEA|ENSG00000172915.20|EH38E1667609|13-35636102-G-A',
 'cardiac_neuro_cava_random:ALT_CFAP91|ENSG00000183833.16|EH38E3533715_fwd_tile1-1_CFAP91|ENSG00000183833.16|EH38E3533715|3-119665026-T-G',
 'cardiac_neuro_cava_random:ALT_TCF20|ENSG00000100207.21|EH38E3484401_rev_tile1-1_TCF20|ENSG00000100207.21|EH38E3484401|22-42381710-G-A',
 'cardiac_neuro_cava_random:ALT_SYNGAP1|ENSG00000197283.18|EH38E3702084_fwd_tile1-1_SYNGAP1|

In [34]:
# sort df by name
replicate_1_assignment.sort_values(by='name')

,barcode,rep1_dna_counts,rep1_rna_counts,name,assignment_number
1764659,CTCGTGATGAAGCGG,3,7,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,16/16
1672917,CGTTCTACCCGTTTA,10,10,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,23/26
2521186,GGACAATGACCGCCG,3,15,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,12/12
516158,AGACTGAACGCTAGG,8,14,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,24/24
3575377,TCGAAGGACGGGGGC,2,1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,5/6
...,...,...,...,...,...
1916051,GAAAGTAGCCTCCTT,3,6,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,12/14
1739567,CTCAGGTTACCAAGT,4,5,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,7/7
4260902,TTTCGACCGGATGCG,3,8,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,9/9
611673,AGCTGAGGTGCGCAA,1,1,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,8/8


In [2]:
# merged data from MPRAsnakeflow
mpra_results_minThreshold = pd.read_csv(config['files']['final_design']['merged_mprasnakeflow_final_resequencing'], sep='\t')
mpra_results_minThreshold

,condition,replicate,name,dna_counts,rna_counts,dna_normalized,rna_normalized,ratio,log2,n_obs_bc
0,NGN2,1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,455,1333,0.117015,0.113946,1.003588,0.005167,112
1,NGN2,1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,428,2869,0.122060,0.271955,2.296273,1.199294,101
2,NGN2,1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,286,874,0.117684,0.119537,1.046844,0.066047,70
3,NGN2,1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,359,1189,0.108848,0.119825,1.134551,0.182122,95
4,NGN2,1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,235,736,0.123071,0.128116,1.072869,0.101474,55
...,...,...,...,...,...,...,...,...,...,...
206005,NGN2,3,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,280,1571,0.092946,0.188011,2.106237,1.074668,83
206006,NGN2,3,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,102,357,0.100368,0.126647,1.313885,0.393838,28
206007,NGN2,3,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,96,172,0.073472,0.047458,0.672584,-0.572214,36
206008,NGN2,3,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,449,1048,0.105733,0.088974,0.876202,-0.190665,117


,0,1,2,3
0,AAAAAAAAAAACGTC,GC_Vista:eye;fb;hb;mb_hs1644_vistaElementContr...,15;270M;NM:i:0;MD:Z:270;60,5/5
1,AAAAAAAAAACAAGT,cardiac_neuro_cava_random:ALT_ZBTB20|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,6/7
2,AAAAAAAAAACACCA,cardiac_neuro_cava_random:ALT_FTO|ENSG00000140...,15;270M;NM:i:0;MD:Z:270;6,9/10
3,AAAAAAAAAACCTCG,cardiac_neuro_cava_random:REF_FKRP|ENSG0000018...,15;270M;NM:i:0;MD:Z:270;6,7/7
4,AAAAAAAAAAGCTGG,cardiac_neuro_cava_random:ALT_RAD51B|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,7/8
...,...,...,...,...
6305461,TTTTTTTTTTGACCG,cardiac_neuro_cava_random:ALT_CPS1|ENSG0000002...,15;270M;NM:i:0;MD:Z:270;6,32/33
6305462,TTTTTTTTTTGACGA,cardiac_neuro_cava_random:ALT_ACTN2|ENSG000000...,15;270M;NM:i:0;MD:Z:270;6,11/11
6305463,TTTTTTTTTTGCACA,cardiac_neuro_cava_random:ALT_CARD11|ENSG00000...,15;270M;NM:i:0;MD:Z:270;6,22/26
6305464,TTTTTTTTTTGCTAA,cardiac_neuro_cava_random:REF_NR4A2|ENSG000001...,15;270M;NM:i:0;MD:Z:270;6,6/6
